# NestedSimPy — Dual Sourcing with Lookahead Expediting

This notebook runs a NestedSimPy-specific example: the dual-sourcing inventory model of Song, Xiao, Zhang and Zipkin (2017), where lead times are endogenous and each review decides how many units to expedite (the two-argument `fn(state, action)` decision form). It uses `env.decide` and `set_inner_actions`.

See the [example page](https://nestedsimpy.github.io/official-parity/dual-sourcing.html) for the side-by-side plain/nested code.

## 1. Install

_Pre-release: NestedSimPy installs from a hosted wheel. (After the public release this becomes `pip install nestedsimpy`.)_

In [ ]:
# Pre-release install from a hosted wheel (Google Drive).
!pip install -q gdown
import gdown
gdown.download(id="1N7mlgDVpVids6Ekr4p2e-gEUrUodiEuq",
               output="nestedsimpy-0.1.0-py3-none-any.whl", quiet=True)
!pip install -q "nestedsimpy-0.1.0-py3-none-any.whl[plot]"

import nestedsimpy
print("NestedSimPy ready —", len(nestedsimpy.__all__), "public objects")

## 2. Run the nested example

The model is written to a file and run as a subprocess; the output below is the **outer** trajectory. In rollout mode it executes the lookahead picks, so it can differ from the plain example.

In [ ]:
%%writefile dual_sourcing_colab.py
# --- inline prelude (replaces the examples' local _imports shim) ---
import argparse, os, random, shutil, sys, itertools
from pathlib import Path

import simpy
import nestedsimpy
from nestedsimpy import (
    NestedEnvironment, NestedResource, NestedPreemptiveResource,
    NestedStore, NestedContainer,
)
try:
    from nestedsimpy.postprocess import (
        package_latest_run, relocate_raw_artifacts, export_realizations,
    )
except Exception:  # pragma: no cover
    package_latest_run = relocate_raw_artifacts = export_realizations = None

DEFAULT_OUT_ROOT = Path("nested_output")
DEFAULT_AUTOPLOT = False
REPO_ROOT = Path(".")
PACKAGE_ROOT = Path(".")

def default_out(*parts):
    p = DEFAULT_OUT_ROOT.joinpath(*map(str, parts)); p.mkdir(parents=True, exist_ok=True); return p

def set_nested_output_folder(*parts):
    p = Path(os.path.join(*[str(x) for x in parts])); p.mkdir(parents=True, exist_ok=True); return p
# --- end prelude ---

"""
Dual-sourcing inventory example -- lookahead expediting.

Covers:

- Continuous review with endogenous lead times (orders queue in a
  production line, so ordering more lengthens the lead times)
- Resources: Resource (two single-server production stages in tandem)

Scenario:
  The dual-sourcing model of Song, Xiao, Zhang and Zipkin (2017),
  "Optimal Policies for a Dual-Sourcing Inventory Problem with
  Endogenous Stochastic Leadtimes", Operations Research 65(2):379-395.
  A single product faces unit Poisson demand with full backlogging.
  The regular supply channel is a two-stage tandem production line
  (one unit at a time, exponential service at each stage), so lead
  times are endogenous: ordering more congests the line and lengthens
  them, and orders never cross. An expedited order skips stage 1 and
  joins stage 2 directly, for a premium per unit. The policy here is
  single sourcing: after every demand and every delivery, top the
  inventory position up to S_REG with regular orders, never expedite
  -- so every unit rides the congested two-stage line.
  This is the lookahead version: at each demand epoch (and once at
  t=0), env.decide tries each expedite count in inner simulations
  launched from the live production line and executes the best one.
"""


from dataclasses import dataclass

import numpy as np

RANDOM_SEED = 2024
DEMAND_RATE = 5.0        # Poisson demand (units per unit time)
STAGE1_RATE = 6.0        # exponential production rate, stage 1
STAGE2_RATE = 7.0        # exponential production rate, stage 2
HOLD_COST = 1.0          # per unit on hand per unit time
BACKLOG_COST = 9.0       # per unit backlogged per unit time
EXPEDITE_PREMIUM = 15.0  # extra cost per expedited unit
HORIZON = 30.0           # length of one run
INIT_NET = 10            # on-hand stock at time 0, empty pipeline
S_REG = 10               # order up to S_REG on the position (regular)

ACTIONS = [None, 1, 2]   # units to expedite now; None = the base rule
INNER_HORIZON = 4.0      # lookahead window, in time units
INNER_REPS = 12          # replications per action

DEMAND_DIST = {"distribution": "exponential", "rate": DEMAND_RATE}
STAGE1_DIST = {"distribution": "exponential", "rate": STAGE1_RATE}
STAGE2_DIST = {"distribution": "exponential", "rate": STAGE2_RATE}

NESTED_OUTPUT_FOLDER = set_nested_output_folder("simpy_examples",
                                                "dual_sourcing")


@dataclass
class State:
    net: int      # on-hand minus backlog (negative = backlogged)
    stage1: int   # units waiting at or in service at stage 1
    stage2: int   # units waiting at or in service at stage 2

    @property
    def position(self) -> int:
        return self.net + self.stage1 + self.stage2


def base_policy(state):
    """Single sourcing: top the position up to S_REG, never expedite."""
    return max(0, S_REG - state.position), 0


def lookahead_policy(state, action):
    """Complete the assigned action with the model's own coupling rule.

    The two decision variables are coupled (regular tops up to S_REG
    AFTER the expedite count is known), so a bare action cannot say
    both -- this policy takes the two-argument form and completes the
    decision itself. The action is the number of units to expedite
    now; None is the base rule's own decision."""
    if action is None:
        return base_policy(state)
    expedited = int(action)
    regular = max(0, S_REG - state.position - expedited)
    return regular, expedited


def accrue(env, state, costs, last_accrual):
    """Charge holding/backlog cost since the last change of net."""
    dt = env.now - last_accrual[0]
    increment = (HOLD_COST * max(state.net, 0)
                 + BACKLOG_COST * max(-state.net, 0)) * dt
    costs["holding"] += HOLD_COST * max(state.net, 0) * dt
    costs["backorder"] += BACKLOG_COST * max(-state.net, 0) * dt
    if increment:
        env.record("cost", increment)               # scores the branches
    last_accrual[0] = env.now


def produced_unit(env, sim, expedited):
    """One ordered unit's life until it reaches inventory."""
    state = sim["state"]
    if not expedited:
        with sim["stage1"].request() as turn:
            yield turn
            yield env.nested_timeout(STAGE1_DIST)
        state.stage1 -= 1
        state.stage2 += 1
    with sim["stage2"].request() as turn:
        yield turn
        yield env.nested_timeout(STAGE2_DIST)
    state.stage2 -= 1
    accrue(env, state, sim["costs"], sim["last_accrual"])
    state.net += 1                                  # delivery
    yield from review(env, sim, decide=False)


def review(env, sim, decide):
    """Consult the policy and launch its orders into the supply system."""
    state = sim["state"]
    if decide:
        regular, expedited = yield from env.decide(lookahead_policy, state)
    else:
        regular, expedited = base_policy(state)
    sim["counts"]["regular"] += regular
    sim["counts"]["expedited"] += expedited
    if expedited:
        premium = expedited * EXPEDITE_PREMIUM
        sim["costs"]["ordering"] += premium
        env.record("cost", premium)
    # Pipeline counts are updated at order time, so any later review at
    # the same instant already sees these orders.
    for _ in range(expedited):
        state.stage2 += 1
        env.process(produced_unit(env, sim, expedited=True))
    for _ in range(regular):
        state.stage1 += 1
        env.process(produced_unit(env, sim, expedited=False))


def demand_process(env, sim):
    while True:
        yield env.nested_timeout(DEMAND_DIST)
        state = sim["state"]
        accrue(env, state, sim["costs"], sim["last_accrual"])
        state.net -= 1                              # backlog if negative
        yield from review(env, sim, decide=True)


def run():
    np.random.seed(RANDOM_SEED)
    env = NestedEnvironment()
    sim = {
        "state": State(net=INIT_NET, stage1=0, stage2=0),
        "stage1": NestedResource(env, capacity=1, nested_id="stage1"),
        "stage2": NestedResource(env, capacity=1, nested_id="stage2"),
        "costs": {"holding": 0.0, "backorder": 0.0, "ordering": 0.0},
        "counts": {"regular": 0, "expedited": 0},
        "last_accrual": [0.0],
    }
    env.process(demand_process(env, sim))
    env.process(review(env, sim, decide=True))   # initial decision at t=0

    # No trigger configuration: NestedSimPy branches on decide's event.
    env.set_outer_stopping_condition(timeout=HORIZON)
    env.set_inner_stopping_condition(relative_time=INNER_HORIZON)
    env.set_inner_repetitions(INNER_REPS)
    env.set_rng("independent")
    env.set_outer_seed(RANDOM_SEED)
    env.set_inner_actions(ACTIONS, metric="cost", outer_run_mode="rollout")
    env.set_output_options(out_dir=NESTED_OUTPUT_FOLDER, gzip_trace=False)
    env.nested_run()
    accrue(env, sim["state"], sim["costs"], sim["last_accrual"])
    total = sum(sim["costs"].values())
    return total, sim, env


if __name__ == "__main__":
    total, sim, env = run()
    print(f"rollout over {len(ACTIONS)} expedite levels: "
          f"total cost {total:.1f} over {HORIZON:.0f} "
          f"({total / HORIZON:.2f} per unit time)")
    print(f"  ordered {sim['counts']['regular']} regular, "
          f"{sim['counts']['expedited']} expedited; ending net "
          f"{sim['state'].net}, in line {sim['state'].stage1}+"
          f"{sim['state'].stage2}")


In [ ]:
# Run as a subprocess so the outer output is clean (inner branches run in separate processes).
!python dual_sourcing_colab.py

## 3. Inspect the run

The lookahead CSVs are read back with pandas: the executed picks and the per-action score table.

In [ ]:
import glob, os
import pandas as pd

run = os.path.dirname(glob.glob("simpy_examples/inventory_lookahead/**/rollout", recursive=True)[0])

# The four rollout CSVs, coarsest to finest: per-action scores, the picks,
# one row per inner simulation, every decision inside every branch.
picks = pd.read_csv(f"{run}/rollout/picks.csv")
actions = pd.read_csv(f"{run}/rollout/actions.csv")
print(picks.to_string(index=False))
print()
print(actions.head(8).to_string(index=False))  # an empty action cell is the base policy
